<a href="https://colab.research.google.com/github/anushka-1318/data-science/blob/main/bfs_dfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Breadth-First Search (BFS)

BFS is an algorithm for traversing or searching tree or graph data structures. It starts at the tree root (or some arbitrary node of a graph, sometimes referred to as a 'search key') and explores all of the neighbor nodes at the present depth prior to moving on to the nodes at the next depth level.

### Implementation Details:
*   It uses a **queue** data structure to keep track of the nodes to visit.
*   It visits nodes level by level.

In [1]:
from collections import deque

def bfs(graph, start_node):
    visited = set()
    queue = deque([start_node])
    visited.add(start_node)
    traversal_order = []

    while queue:
        current_node = queue.popleft()
        traversal_order.append(current_node)

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return traversal_order

# Example Graph (Adjacency List)
graph_bfs = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

print("BFS Traversal (starting from A):")
print(bfs(graph_bfs, 'A'))

BFS Traversal (starting from A):
['A', 'B', 'C', 'D', 'E', 'F']


## Depth-First Search (DFS)

DFS is an algorithm for traversing or searching tree or graph data structures. The algorithm starts at the root node (selecting some arbitrary node as the root node in the case of a graph) and explores as far as possible along each branch before backtracking.

### Implementation Details:
*   It typically uses **recursion** or an explicit **stack** data structure.
*   It explores as deep as possible along each path.

In [2]:
def dfs(graph, start_node, visited=None, traversal_order=None):
    if visited is None:
        visited = set()
    if traversal_order is None:
        traversal_order = []

    visited.add(start_node)
    traversal_order.append(start_node)

    for neighbor in graph[start_node]:
        if neighbor not in visited:
            dfs(graph, neighbor, visited, traversal_order)
    return traversal_order

# Example Graph (Adjacency List) - using the same graph for consistency
graph_dfs = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

print("DFS Traversal (starting from A):")
print(dfs(graph_dfs, 'A'))

DFS Traversal (starting from A):
['A', 'B', 'D', 'E', 'F', 'C']


## A* Search Algorithm

A* is a popular pathfinding algorithm used in various applications, including game development and robotics. It's an informed search algorithm, meaning it uses a heuristic function to guide its search.

### Key Concepts:
*   **g-score**: The cost from the start node to the current node.
*   **h-score (Heuristic)**: The estimated cost from the current node to the goal node. This must be an *admissible* heuristic (never overestimates the true cost).
*   **f-score**: The sum of g-score and h-score (`f = g + h`). A* prioritizes nodes with the lowest f-score.
*   **Open Set**: A list of nodes to be evaluated, typically implemented as a priority queue sorted by f-score.
*   **Closed Set**: A list of nodes already evaluated.

In [4]:
import heapq

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0  # Cost from start to current node
        self.h = 0  # Heuristic (estimated cost from current to goal)
        self.f = 0  # Total cost (g + h)

    # Override comparison methods for priority queue (heapq)
    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.position == other.position

    def __hash__(self):
        return hash(self.position)


def heuristic(a, b):
    """Manhattan distance heuristic for a grid"""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar_search(grid, start, end):
    # Create start and end nodes
    start_node = Node(start)
    end_node = Node(end)

    open_set = []  # Priority queue for nodes to evaluate
    heapq.heappush(open_set, start_node)

    closed_set = set()  # Set of nodes already evaluated

    while open_set:
        current_node = heapq.heappop(open_set)

        if current_node == end_node:
            path = []
            current = current_node
            while current is not None:
                path.append(current.position)
                current = current.parent
            return path[::-1]  # Return reversed path

        closed_set.add(current_node)

        # Explore neighbors
        # Possible movements: Up, Down, Left, Right
        for new_position in [(0, -1), (0, 1), (-1, 0), (1, 0)]:
            node_position = (current_node.position[0] + new_position[0],
                             current_node.position[1] + new_position[1])

            # Make sure neighbor is within grid bounds
            if (node_position[0] < 0 or node_position[0] >= len(grid) or
                node_position[1] < 0 or node_position[1] >= len(grid[0])):
                continue

            # Make sure neighbor is not an obstacle (represented by 1 in grid)
            if grid[node_position[0]][node_position[1]] == 1:
                continue

            neighbor = Node(node_position, current_node)

            if neighbor in closed_set:
                continue

            # Calculate g, h, and f values
            neighbor.g = current_node.g + 1 # Assuming cost of 1 per step
            neighbor.h = heuristic(neighbor.position, end_node.position)
            neighbor.f = neighbor.g + neighbor.h

            # Check if neighbor is already in open_set with a higher g_score
            # If so, update it. Otherwise, add to open_set.
            found_in_open = False
            for open_node in open_set:
                if neighbor == open_node and neighbor.g >= open_node.g:
                    found_in_open = True
                    break

            if not found_in_open:
                heapq.heappush(open_set, neighbor)

    return None # No path found

# Example Grid:
# 0 = walkable, 1 = obstacle
grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

start_point = (0, 0)
end_point = (9, 9)

print(f"Searching for path from {start_point} to {end_point}:")
path = astar_search(grid, start_point, end_point)

if path:
    print("Path found:")
    for p in path:
        print(p)

    # Visualize the path on the grid (optional)
    grid_copy = [row[:] for row in grid] # Create a copy to modify
    for x, y in path:
        grid_copy[x][y] = 'P' # Mark path
    grid_copy[start_point[0]][start_point[1]] = 'S'
    grid_copy[end_point[0]][end_point[1]] = 'E'

    print("\nGrid with Path:")
    for row in grid_copy:
        print(' '.join(map(str, row)))
else:
    print("No path found!")

Searching for path from (0, 0) to (9, 9):
Path found:
(0, 0)
(0, 1)
(0, 2)
(0, 3)
(1, 3)
(2, 3)
(3, 3)
(4, 3)
(5, 3)
(5, 4)
(5, 5)
(5, 6)
(6, 6)
(7, 6)
(7, 7)
(7, 8)
(8, 8)
(9, 8)
(9, 9)

Grid with Path:
S P P P 1 0 0 0 0 0
0 0 0 P 1 0 0 0 0 0
0 0 0 P 1 0 0 0 0 0
0 0 0 P 1 0 0 0 0 0
0 0 0 P 1 0 0 0 0 0
0 0 0 P P P P 0 0 0
0 0 0 0 1 0 P 0 0 0
0 0 0 0 1 0 P P P 0
0 0 0 0 1 0 0 0 P 0
0 0 0 0 0 0 0 0 P E


## Minimax Algorithm

The Minimax algorithm is a decision-making algorithm used in game theory, artificial intelligence, and operations research for minimizing the possible loss for a worst-case (maximum loss) scenario. It's typically used for two-player turn-based games with perfect information.

### Key Concepts:
*   **Players**: Maximizing player (AI) and Minimizing player (opponent).
*   **Game Tree**: Represents all possible moves and outcomes of a game.
*   **Utility Function**: Evaluates the outcome of a terminal state (win, loss, draw).
*   **Recursion**: The algorithm works by recursively exploring the game tree.
*   **Maximizing Player**: Chooses the move that leads to the highest possible utility.
*   **Minimizing Player**: Chooses the move that leads to the lowest possible utility for the maximizing player.

In [6]:
def minimax(node, depth, maximizing_player):
    # Base case: If node is a terminal node or depth limit is reached
    # In this simple example, terminal nodes are integers (scores)
    if isinstance(node, int):
        return node

    # If we have reached the depth limit and it's not a terminal node,
    # we need a heuristic evaluation function. For simplicity here, we assume
    # all leaf nodes are terminal or we've simplified the tree.
    if depth == 0: # This condition might be used if nodes were more complex
        # In a real game, this would be a heuristic evaluation
        return sum(minimax(child, depth, not maximizing_player) for child in node) / len(node)

    if maximizing_player:
        max_eval = -float('inf')
        for child in node:
            eval = minimax(child, depth - 1, False)
            max_eval = max(max_eval, eval)
        return max_eval
    else:
        min_eval = float('inf')
        for child in node:
            eval = minimax(child, depth - 1, True)
            min_eval = min(min_eval, eval)
        return min_eval

# Example Game Tree (represented as nested lists)
# Numbers represent scores at terminal nodes (from MAX player's perspective)
# A higher number is better for the maximizing player

# Level 0 (Root - MAX)
#   Level 1 (MIN)
#     Level 2 (MAX)
#       Level 3 (Terminal scores)

game_tree = [
    [
        [3, 5],    # MAX chooses 5
        [2, 9]     # MAX chooses 9
    ],             # MIN chooses the branch that leads to lower score for MAX (e.g., 5 vs 9)
    [
        [1, 0],    # MAX chooses 1
        [4, 8]     # MAX chooses 8
    ]              # MIN chooses the branch that leads to lower score for MAX (e.g., 1 vs 8)
]

# Let's add another branch to make it slightly more complex
game_tree_complex = [
    [
        [3, 5],     # MAX chooses 5
        [2, 9]      # MAX chooses 9
    ],
    [
        [1, 0],     # MAX chooses 1
        [4, 8]      # MAX chooses 8
    ],
    [
        [7, 1],     # MAX chooses 7
        [6, 10]     # MAX chooses 10
    ]
]


print("Minimax result for simple game tree:")
# Start with depth = 3 (number of levels in the tree before terminal nodes)
# and maximizing_player = True (since the root is always MAX's turn)
result_simple = minimax(game_tree, 3, True)
print(f"The optimal value for the maximizing player is: {result_simple}")

print("\nMinimax result for complex game tree:")
result_complex = minimax(game_tree_complex, 3, True)
print(f"The optimal value for the maximizing player is: {result_complex}")

Minimax result for simple game tree:
The optimal value for the maximizing player is: 5

Minimax result for complex game tree:
The optimal value for the maximizing player is: 7
